# DQ-01 · customers.csv
Checks: null rate, duplicates, FK → geography, temporal, domain values.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── helpers ───────────────────────────────────────────────────────────────────
issues = []

def flag(label, mask_or_count, df=None, show_sample=True):
    count = int(mask_or_count.sum()) if hasattr(mask_or_count, 'sum') else int(mask_or_count)
    total = len(df) if df is not None else None
    pct   = f' ({count/total*100:.2f}%)' if total else ''
    status = '❌ ISSUE' if count > 0 else '✅ OK'
    print(f'{status}  {label}: {count:,}{pct}')
    if count > 0:
        issues.append(label)
        if show_sample and df is not None and hasattr(mask_or_count, 'sum'):
            sample = df[mask_or_count].head(3)
            print(sample.to_string(index=False))
    return count

def summary():
    print()
    if issues:
        print(f'══ {len(issues)} issue(s) found ══')
        for i in issues: print(f'  • {i}')
    else:
        print('══ All checks passed ══')


In [ ]:
cust = pd.read_csv('customers.csv', parse_dates=['signup_date'])
geo  = pd.read_csv('geography.csv')
print(f'Shape: {cust.shape}')
cust.head(3)

## 1. Null rate

In [ ]:
null_counts = cust.isnull().sum()
null_pct    = null_counts / len(cust) * 100
print(pd.DataFrame({'null_count': null_counts, 'null_%': null_pct.round(2)}))

## 2. Duplicate customer_id

In [ ]:
flag('Duplicate customer_id', cust.duplicated(subset='customer_id'), cust)

## 3. FK: zip → geography.zip

In [ ]:
valid_zips  = set(geo['zip'])
bad_zip     = ~cust['zip'].isin(valid_zips)
flag('customer zip not in geography', bad_zip, cust)

## 4. City consistency with zip

In [ ]:
zip_city_map = geo.set_index('zip')['city'].to_dict()
cust['expected_city'] = cust['zip'].map(zip_city_map)
mismatch = cust['city'] != cust['expected_city']
flag('city ≠ geography.city for same zip', mismatch, cust)
cust.drop(columns='expected_city', inplace=True)

## 5. Domain values

In [ ]:
VALID_GENDER  = {'Male','Female','Non-binary'}
VALID_AGE     = {'18-24','25-34','35-44','45-54','55+'}
VALID_CHANNEL = {'organic_search','social_media','paid_search','email_campaign','referral','direct'}

flag('Invalid gender',              ~cust['gender'].isin(VALID_GENDER),  cust)
flag('Invalid age_group',           ~cust['age_group'].isin(VALID_AGE),  cust)
flag('Invalid acquisition_channel', ~cust['acquisition_channel'].isin(VALID_CHANNEL), cust)

## 6. Temporal: signup_date sanity

In [ ]:
flag('signup_date is null',           cust['signup_date'].isnull(),              cust)
flag('signup_date before 2000',       cust['signup_date'].dt.year < 2000,        cust)
flag('signup_date after 2022-12-31',  cust['signup_date'] > pd.Timestamp('2022-12-31'), cust)

## Summary

In [ ]:
summary()